# Demographics Break Down

In [ ]:
import os
os.chdir("../")

from argparse import Namespace

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

from data import *
from run import *

data_path = "../excel_data"

## Get Valid Samples

Rebuild the datasets as done in the code to get an accurate representation of used samples

In [ ]:
args = Namespace(
    seed=0, 
    data_path="../${model}_multimodal_bins",
    test_path="../${model}_multimodal_bins_rw",
    clinical=True,
    path_lang=True,
    rad_lang=True,
    label_col="survival_days"
)

rih, rw = get_datasets(args) #
rih_loader, rw_loader = DataLoader(rih), DataLoader(rw)
#dict_keys(['label', 'label_mask', 'path_lang_mask', 'path_lang', 'rad_lang_mask', 'rad_lang', 'clinical_mask', 'clinical', 'slide_ids', 'survival_days', 'survival_right_censor'])
rih_df, rw_df = pd.readexcel(f"{data_path}/PDAC IHC.xlsx"), pd.readexcel(f"{data_path}/Combined_Clinical_lab.xlsx")

In [ ]:
#for each sample in loader, need to pull the id and survival time
rih_ids, rih_surv   = [], []
rw_ids, rw_surv     = [], []

for sample in rih_loader:
    rih_ids.append(sample['slide_ids'].item())
    rih_surv.append(sample['survival_days'].item())

rih_df = rih_df[rih_df['PDAC ID'] in rih_ids]
rw_df = rw_df[rw_df['Patient ID'] in rw_ids]

In [ ]:
def desc_cat_column(df, col, mapping=None, mapping_name = None):
    """
    
    df: pandas dataframe
    col: column in df to summarize
    mapping: map the unique values in col (dict)
    mapping_name: name of mapped column
    """
    desc = pd.concat([df[col].value_counts(), df[col].value_counts(normalize=True)], axis=1)
    if mapping is not None:
        desc[mapping_name] = desc.index.to_series().map(mapping)
        desc = desc[[mapping_name, 'count', 'proportion']]

    return desc

#waht are the covariates relevant here?
#i guess we can do like age, gender, survival time, tumor grade 

#survival time: just grab from dataloader

#RIH DATA: "PDAC IHC.xlsx"
#age: 'Age at Diagnosis'
#gender: 'Sex'
#tumor grade: 'AJCC Pathologic Stage Group'

#RW DATA: Combined_Clinical_lab.xlsx -> make sure to re-export this
#age: 'Age'
#gender: '''Sex \nFemale=0\nMale=1'''
#tumor grade: 'AJCC Pathologic Stage Group'

### Age

In [ ]:
print(rih_df['Age at Diagnosis'].describe())
print(rw_df['Age'].describe())

In [ ]:
print(desc_cat_column(rih_df, 'Sex'))
print(desc_cat_column(rw_df, 'Sex \nFemale=0\nMale=1', {0: "Female", 1: "Male"}, 'Sex'))

In [ ]:
print(desc_cat_column(rih_df, 'AJCC Pathologic Stage Group'))
print(desc_cat_column(rw_df, 'AJCC Pathologic Stage Group'))
